In [1]:
# Cell 1: Imports (Updated for SQLAlchemy)
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
import re
from datetime import datetime, timedelta

# Cấu hình để Pandas hiển thị đầy đủ nội dung
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("Cell 1: Các thư viện cần thiết đã được import.")

Cell 1: Các thư viện cần thiết đã được import.


In [2]:
# Cell 2: Load Data from PostgreSQL (Your improved SQLAlchemy version)

load_dotenv()
DB_TABLE_NAME = 'topcv_jobs_detailed'

print(f"\nCell 2: Đang tải dữ liệu từ bảng '{DB_TABLE_NAME}'...")

df_jobs = pd.DataFrame()

try:
    # Đọc thông tin kết nối từ biến môi trường
    dbname = os.getenv('DB_NAME')
    user = os.getenv('DB_USER')
    password = quote_plus(os.getenv('DB_PASSWORD') or '')
    host = os.getenv('DB_HOST', 'localhost')
    port = os.getenv('DB_PORT', '5432')

    # Tạo SQLAlchemy engine
    engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}")

    # Đọc dữ liệu
    with engine.connect() as connection:
        query = f"SELECT * FROM {DB_TABLE_NAME} WHERE status = 'completed';"
        df_jobs = pd.read_sql(query, connection)

    if not df_jobs.empty:
        print(f"  => Đã tải thành công {len(df_jobs)} job items (status='completed').")

        # Xử lý các cột dạng array
        array_cols = ['related_job_categories', 'required_skills_tags', 'preferred_skills_tags']
        for col in array_cols:
            if col in df_jobs.columns:
                df_jobs[col] = df_jobs[col].apply(lambda x: x if x is not None and isinstance(x, list) else [])
    else:
        print("  Cảnh báo: Không có dữ liệu (status='completed') trong DB để phân tích. Hãy đảm bảo script `detail_worker.py` đang chạy.")

except Exception as e:
    print(f"  Lỗi khi tải dữ liệu từ DB: {e}")


Cell 2: Đang tải dữ liệu từ bảng 'topcv_jobs_detailed'...
  => Đã tải thành công 50 job items (status='completed').


In [3]:
# Cell 3: Data Overview & Initial Inspection

if not df_jobs.empty:
    print("--- (1/3) Thông tin tổng quan (df_jobs.info()) ---")
    df_jobs.info()
    
    print("\n--- (2/3) 5 dòng dữ liệu đầu tiên (df_jobs.head()) ---")
    display(df_jobs.head())
    
    print("\n--- (3/3) Thống kê các giá trị rỗng (df_jobs.isnull().sum()) ---")
    # Hiển thị tất cả các dòng, sắp xếp theo số lượng null giảm dần
    with pd.option_context('display.max_rows', None):
        display(df_jobs.isnull().sum().sort_values(ascending=False))
else:
    print("Cell 3: DataFrame rỗng, không có gì để hiển thị.")

--- (1/3) Thông tin tổng quan (df_jobs.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   job_id                     50 non-null     int64         
 1   job_title                  50 non-null     object        
 2   job_url                    50 non-null     object        
 3   scrape_date                50 non-null     object        
 4   status                     50 non-null     object        
 5   company_name_raw_list      50 non-null     object        
 6   salary_raw_list            50 non-null     object        
 7   location_raw_list          50 non-null     object        
 8   post_date_raw_list         50 non-null     object        
 9   company_name_detail        50 non-null     object        
 10  company_scale              50 non-null     object        
 11  company_field         

,job_id,job_title,job_url,scrape_date,status,company_name_raw_list,salary_raw_list,location_raw_list,post_date_raw_list,company_name_detail,company_scale,company_field,company_full_address,job_level,education_level,quantity_needed,employment_type,gender_requirement,related_job_categories,required_skills_tags,preferred_skills_tags,job_description_text,job_requirements_text,job_benefits_text,working_time_text,application_deadline_date,inserted_at,updated_at
0,5,Chuyên Viên Kinh Doanh/ Phát Triển Thị Trường ...,https://www.topcv.vn/viec-lam/chuyen-vien-kinh...,2025-06-23,completed,Công ty TNHH Thiết bị phụ tùng An Phát,15 - 20 triệu,Hồ Chí Minh,5 ngày trước,Công ty TNHH Thiết bị phụ tùng An Phát,100-499 nhân viên,Bảo trì / Sửa chữa,"Số 8 Trung Yên 3, Trung Hòa, Cầu Giấy, Hà Nội",Nhân viên,Cao Đẳng trở lên,1 người,Toàn thời gian,Nam,"[Kinh doanh/Bán hàng, Sản xuất, Sales Sản xuất...","[Kỹ năng giao tiếp, Kỹ năng đàm phán, Kỹ năng ...","[Kỹ Năng Tìm Kiếm Khách Hàng Tiềm Năng, Am Hiể...","- Phát triển hệ thống kênh phân phối, đại lý t...","- Giới tính: Nam, tuổi từ 25 – 35\n- Tốt nghiệ...",- Thu nhập hấp dẫn chưa bao gồm:\n+ Thưởng:Thư...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:00)\nNghỉ 02 Th...,18/07/2025,2025-06-23 22:57:43.674444,2025-06-24 11:14:15.307026
1,9,Nhân Viên Kinh Doanh B2B - (Bắt Buộc Từ 03 - 0...,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,2025-06-23,completed,CÔNG TY CỔ PHẦN GIẢI PHÁP ĐIỆN TỬ VIỆT HỒNG QUANG,Thoả thuận,Hồ Chí Minh,1 tuần trước,CÔNG TY CỔ PHẦN GIẢI PHÁP ĐIỆN TỬ VIỆT HỒNG QUANG,25-99 nhân viên,IT - Phần mềm,"tầng 2 số 328 đường Võ Văn Kiệt, P.Cô Giang, Q...",Nhân viên,Đại Học trở lên,3 người,Toàn thời gian,None,"[Kinh doanh/Bán hàng, Công nghệ Thông tin, Sal...",[],[],"- Tìm hiểu nhu cầu của khách hàng,phối hợρ phò...",- Tuổi từ 23-30 hoặc 03- 05 năm kinh nghiệm vị...,- Lương cứng + Hoa Hồng\n- Được training thêm...,Thứ 2 - Thứ 6 (từ 08:00 đến 17:00),10/07/2025,2025-06-23 22:57:43.679056,2025-06-24 11:16:51.118026
2,13,Tài Xế Lái Xe Văn Phòng (Nam) - Từ 05 Năm Kinh...,https://www.topcv.vn/viec-lam/tai-xe-lai-xe-va...,2025-06-23,completed,CÔNG TY CỔ PHẦN GIẢI PHÁP ĐIỆN TỬ VIỆT HỒNG QUANG,10 - 15 triệu,Hồ Chí Minh,2 tuần trước,CÔNG TY CỔ PHẦN GIẢI PHÁP ĐIỆN TỬ VIỆT HỒNG QUANG,25-99 nhân viên,IT - Phần mềm,"tầng 2 số 328 đường Võ Văn Kiệt, P.Cô Giang, Q...",Nhân viên,Trung cấp trở lên,1 người,Toàn thời gian,Nam,"[Bán lẻ/Dịch vụ đời sống, Tài xế, Tài xế/Lái x...",[Bằng Lái Xe B2],[],-\tLái xe an toàn và đúng giờ theo lịch trình ...,"-\tNam, từ 28 - 40 tuổi, không hút thuốc\n-\tC...","- Mức lương: 10,000,000/ tháng (Gross) + công ...",Thứ 2 - Thứ 6 (từ 08:00 đến 17:00)\nThứ 7 (từ ...,10/07/2025,2025-06-23 22:57:43.683420,2025-06-24 11:20:18.031093
3,23,Chuyên Viên Kinh Doanh Không Yêu Cầu Kinh Nghi...,https://www.topcv.vn/viec-lam/chuyen-vien-kinh...,2025-06-23,completed,CÔNG TY TNHH BẤT ĐỘNG SẢN HỒNG ĐỨC LAND,Từ 16 triệu,Hồ Chí Minh,1 tuần trước,CÔNG TY TNHH BẤT ĐỘNG SẢN HỒNG ĐỨC LAND,25-99 nhân viên,Bất động sản,"Văn phòng 02, Tầng 8, Tòa Nhà Pearl plaza, 561...",Nhân viên,Trung cấp trở lên,10 người,Toàn thời gian,None,"[Kinh doanh/Bán hàng, Xây dựng/Bất động sản, S...","[Đam mê kinh doanh, Kiên Trì]",[],"- Lên kịch bản sale, kế hoạch MKT online để đả...",- Tốt nghiệp Trung cấp trở lên các ngành liên ...,"- Lương cứng lên đến 16,000,000đ/tháng + hoa h...",None,13/07/2025,2025-06-23 22:57:43.693439,2025-06-24 11:28:58.647950
4,27,Nhân Viên Kinh Doanh/ Sale Supervisor Lĩnh Vưc...,https://www.topcv.vn/viec-lam/nhan-vien-kinh-d...,2025-06-23,completed,CÔNG TY TNHH Y TẾ QUỐC TẾ MAIA,Thoả thuận,Hồ Chí Minh,2 tuần trước,CÔNG TY TNHH Y TẾ QUỐC TẾ MAIA,25-99 nhân viên,Dược phẩm / Y tế / Công nghệ sinh học,"556 Điện Biên Phủ, Phường 10, Quận 10, TP Hồ C...",Nhân viên,Đại Học trở lên,5 người,Toàn thời gian,None,"[Kinh doanh/Bán hàng, Dược/Y tế/Sức khoẻ/Công ...",[],[],- Tìm kiếm và phát triển khách hàng mới trong ...,"- Tốt nghiệp đại học chuyên ngành Y, Dược, Sin...","- Mức lương cạnh tranh, thưởng theo hiệu quả c..


--- (3/3) Thống kê các giá trị rỗng (df_jobs.isnull().sum()) ---


gender_requirement           39
working_time_text            17
application_deadline_date     1
job_id                        0
status                        0
company_name_raw_list         0
salary_raw_list               0
location_raw_list             0
post_date_raw_list            0
job_title                     0
job_url                       0
scrape_date                   0
company_field                 0
company_scale                 0
company_name_detail           0
company_full_address          0
quantity_needed               0
employment_type               0
education_level               0
job_level                     0
required_skills_tags          0
related_job_categories        0
preferred_skills_tags         0
job_description_text          0
job_benefits_text             0
job_requirements_text         0
inserted_at                   0
updated_at                    0
dtype: int64

In [4]:
# Cell 4: Data Processing Helper Functions (100% FROM YOUR ORIGINAL NOTEBOOK)
# Đây là các hàm em đã xây dựng, được tái sử dụng nguyên vẹn.

import re
import numpy as np
import pandas as pd

# --- Hàm xử lý Lương (từ Cell 11 gốc) ---
def parse_salary_string_for_list_data(salary_text):
    if pd.isna(salary_text): return np.nan, np.nan, 'triệu VNĐ', 'Không xác định'
    text_original, text_lower = str(salary_text), str(salary_text).lower()
    min_sal, max_sal, currency, sal_type = np.nan, np.nan, 'triệu VNĐ', 'Không xác định'

    if "thoả thuận" in text_lower:
        sal_type = "Thoả thuận"; return min_sal, max_sal, currency, sal_type

    if "usd" in text_lower: currency = "USD"; text_extract = re.sub(r'[^0-9.,\\-]', '', text_lower.replace("usd",""))
    elif "triệu" in text_lower: text_extract = re.sub(r'[^0-9.,\\-]', '', text_lower.replace("triệu","").replace("vnd",""))
    else: text_extract = re.sub(r'[^0-9.,\\-]', '', text_lower)
    
    nums_str = re.findall(r'\d+[\.,]?\d*', text_extract)
    nums_float = []
    for ns in nums_str:
        cleaned_ns = ns.replace('.', '').replace(',', '.') if ',' in ns and '.' not in ns[:ns.find(',')] else ns.replace(',', '')
        try: nums_float.append(float(cleaned_ns))
        except ValueError: pass
    nums = sorted(nums_float)

    if not nums: return min_sal, max_sal, currency, sal_type

    if "tới" in text_lower or "upto" in text_lower or ("đến" in text_lower and len(nums) == 1 and "từ" not in text_lower):
        sal_type = "Tối đa"; max_sal = nums[-1] if nums else np.nan
    elif "từ" in text_lower and not any(k in text_lower for k in ["đến", "tới"]) and "-" not in text_original:
        sal_type = "Tối thiểu"; min_sal = nums[0] if nums else np.nan
    elif (("-" in text_original or "đến" in text_lower) and len(nums) >= 2) or len(nums) >= 2:
        sal_type = "Khoảng"
        if len(nums) >= 2: min_sal, max_sal = nums[0], nums[-1]
        elif len(nums) == 1: min_sal = max_sal = nums[0]; sal_type = "Cố định"
        if pd.notna(min_sal) and min_sal == max_sal: sal_type = "Cố định"
    elif len(nums) == 1:
        sal_type = "Cố định"; min_sal = max_sal = nums[0]
    return min_sal, max_sal, currency, sal_type

# --- Hàm xử lý Kinh nghiệm (từ Cell 11 gốc) ---
def standardize_experience_for_list_data(exp_str):
    if pd.isna(exp_str): return 'Không xác định'
    exp_lower = str(exp_str).lower()
    if any(k in exp_lower for k in ["không yêu cầu", "chưa có kinh nghiệm", "no experience", "fresher", "mới tốt nghiệp"]): return "Không yêu cầu"
    if m := re.search(r'(?:dưới|<)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"< {m.group(1)} năm"
    if m := re.search(r'(?:từ\s*)?(\d+)\s*(?:-|đến)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"{m.group(1)}-{m.group(2)} năm"
    if m := re.search(r'^\d+\s*(?:năm|year)', exp_lower): return f"{m.group(0).strip()} năm" # Chỉnh 1 chút để bắt '1 năm'
    if m := re.search(r'(?:trên|>)\s*(\d+)\s*(?:năm|year)', exp_lower): return f"> {m.group(1)} năm"
    return exp_str # Trả về gốc nếu không khớp

# --- Hàm join list thành string (từ Cell 13 gốc) ---
def join_list_elements(data_list):
    if isinstance(data_list, list): return ', '.join(str(item) for item in data_list) if data_list else None
    return str(data_list) if pd.notna(data_list) else None

# --- Danh sách và hàm trích xuất Kỹ năng (từ Cell 13 gốc) ---
MASTER_SKILL_LIST = [
    "python", "sql", "java", "scala", "r", "c++", "c#", "javascript", "typescript", "php", "swift", "kotlin", "go", "ruby", "rust",
    "pandas", "numpy", "scipy", "scikit-learn", "sklearn", "tensorflow", "keras", "pytorch", "opencv", "nltk", "spacy", "gensim",
    "spark", "pyspark", "hadoop", "mapreduce", "hive", "pig", "kafka", "flink", "storm", "airflow", "luigi", "azkaban", "nifi",
    "docker", "kubernetes", "k8s", "openshift", "ansible", "terraform", "jenkins", "git", "cicd", "ci/cd",
    "aws", "azure", "gcp", "cloud", "s3", "ec2", "lambda", "rds", "dynamodb", "redshift", "glue", "emr", "sagemaker", 
    "azure data factory", "azure databricks", "azure synapse", "google bigquery", "google dataflow", "google dataproc",
    "excel", "vba", "power bi", "powerbi", "tableau", "qlik", "qlikview", "qliksense", "google data studio", "superset", "metabase", "looker",
    "ssis", "ssas", "ssrs", "informatica", "talend", "datastage",
    "etl", "elt", "data pipeline", "data modeling", "data modelling", "data warehouse", "dwh", "data lake", "data mart", "olap",
    "data analysis", "data analyst", "business intelligence", "bi", "business analyst", "ba",
    "data visualization", "data mining", "machine learning", "ml", "deep learning", "dl", "artificial intelligence", "ai", "nlp", "natural language processing",
    "statistics", "statistical modeling", "a/b testing", "hypothesis testing",
    "api", "restful api", "graphql", "microservices", "linux", "unix", "bash", "shell scripting",
    "nosql", "mongodb", "cassandra", "redis", "elasticsearch", "neo4j", "agile", "scrum", "kanban",
    "english", "tiếng anh", "giao tiếp", "communication", "presentation", "trình bày", 
    "problem solving", "critical thinking", "analytical skills", "phân tích", "tư duy logic", "teamwork", "làm việc nhóm"
] 

def extract_master_skills(row, skill_list_master):
    text_jd = str(row.get('job_description_text','')) if pd.notna(row.get('job_description_text')) else ''
    text_req = str(row.get('job_requirements_text','')) if pd.notna(row.get('job_requirements_text')) else ''
    text_combined = (text_jd + " " + text_req).lower()
    if not text_combined.strip(): return None
    found_skills = []
    for skill_keyword in skill_list_master:
        pattern = r'\b' + re.escape(skill_keyword.lower()) + r'\b'
        if re.search(pattern, text_combined):
            found_skills.append(skill_keyword) 
    return list(set(found_skills)) if found_skills else []

print("Cell 4: Các hàm helper xử lý dữ liệu gốc đã được định nghĩa.")

Cell 4: Các hàm helper xử lý dữ liệu gốc đã được định nghĩa.


In [5]:
# Cell 5: Apply Processing Functions to Create New Features

if not df_jobs.empty:
    print("--- Bắt đầu quá trình xử lý và tạo cột mới ---")
    
    # Tạo một bản sao để xử lý, giữ lại df_jobs gốc nếu cần
    df_processed = df_jobs.copy()

    # 1. Xử lý Lương (từ salary_raw_list)
    if 'salary_raw_list' in df_processed.columns:
        salary_cols = ['salary_min', 'salary_max', 'salary_currency', 'salary_type']
        df_processed[salary_cols] = df_processed['salary_raw_list'].apply(
            lambda x: pd.Series(parse_salary_string_for_list_data(x), index=salary_cols)
        )
        print("  (1/6) Đã xử lý cột 'salary_raw_list'.")

    # 2. Xử lý Kinh nghiệm (từ experience_raw_list)
    if 'experience_raw_list' in df_processed.columns:
        df_processed['experience_standardized'] = df_processed['experience_raw_list'].apply(standardize_experience_for_list_data)
        print("  (2/6) Đã xử lý cột 'experience_raw_list'.")

    # 3. Chuyển đổi Ngày tháng
    if 'post_date_raw_list' in df_processed.columns and 'scrape_date' in df_processed.columns:
        # Tạm thời để trống, em có thể copy-paste logic phức tạp của mình vào đây
        # Ví dụ:
        df_processed['post_date_calculated'] = pd.to_datetime(df_processed['scrape_date'], errors='coerce')
        print("  (3/6) Đã xử lý 'post_date_raw_list' (cần logic chi tiết hơn).")
    
    if 'application_deadline_date' in df_processed.columns:
        df_processed['deadline_dt'] = pd.to_datetime(df_processed['application_deadline_date'], format='%d/%m/%Y', errors='coerce')
        print("  (4/6) Đã xử lý 'application_deadline_date'.")

    # 4. Trích xuất Kỹ năng từ Text
    text_cols_for_skills = ['job_description_text', 'job_requirements_text']
    if all(c in df_processed.columns for c in text_cols_for_skills):
        df_processed['skills_from_text'] = df_processed.apply(extract_master_skills, args=(MASTER_SKILL_LIST,), axis=1)
        print("  (5/6) Đã trích xuất kỹ năng từ text JD.")

    # 5. Kết hợp tất cả các kỹ năng
    df_processed['skills_all'] = df_processed.apply(
        lambda row: sorted(list(set(
            (row.get('required_skills_tags') or []) + 
            (row.get('preferred_skills_tags') or []) + 
            (row.get('skills_from_text') or [])
        ))), axis=1
    )
    df_processed['skills_all_str'] = df_processed['skills_all'].apply(join_list_elements)
    print("  (6/6) Đã kết hợp tất cả các nguồn kỹ năng.")
    
    print("\n--- Xử lý hoàn tất! ---")
else:
    print("Cell 5: DataFrame rỗng, bỏ qua xử lý.")

--- Bắt đầu quá trình xử lý và tạo cột mới ---
  (1/6) Đã xử lý cột 'salary_raw_list'.
  (3/6) Đã xử lý 'post_date_raw_list' (cần logic chi tiết hơn).
  (4/6) Đã xử lý 'application_deadline_date'.
  (5/6) Đã trích xuất kỹ năng từ text JD.
  (6/6) Đã kết hợp tất cả các nguồn kỹ năng.

--- Xử lý hoàn tất! ---


In [6]:
# Cell 6: Review Processed DataFrame

if 'df_processed' in locals() and not df_processed.empty:
    print("--- Xem lại DataFrame sau khi xử lý ---")
    
    # Các cột mới được tạo ra
    new_cols = [
        'salary_min', 'salary_max', 'salary_currency', 'salary_type',
        'experience_standardized', 'post_date_calculated', 'deadline_dt',
        'skills_from_text', 'skills_all', 'skills_all_str'
    ]
    
    # Lấy các cột quan trọng để xem trước
    preview_cols = ['job_title', 'company_name_detail'] + [col for col in new_cols if col in df_processed.columns]
    
    display(df_processed[preview_cols].head(10))
    
    print("\n--- Thông tin của DataFrame đã xử lý ---")
    df_processed.info()
else:
    print("Cell 6: Không có DataFrame đã xử lý để xem lại.")

--- Xem lại DataFrame sau khi xử lý ---


,job_title,company_name_detail,salary_min,salary_max,salary_currency,salary_type,post_date_calculated,deadline_dt,skills_from_text,skills_all,skills_all_str
0,Chuyên Viên Kinh Doanh/ Phát Triển Thị Trường ...,Công ty TNHH Thiết bị phụ tùng An Phát,15.0,20.0,triệu VNĐ,Khoảng,2025-06-23,2025-07-18,"[phân tích, giao tiếp]","[Am Hiểu Về Các Loại Thiết Bị Sửa Chữa Ô Tô, H...","Am Hiểu Về Các Loại Thiết Bị Sửa Chữa Ô Tô, Hi..."
1,Nhân Viên Kinh Doanh B2B - (Bắt Buộc Từ 03 - 0...,CÔNG TY CỔ PHẦN GIẢI PHÁP ĐIỆN TỬ VIỆT HỒNG QUANG,NaN,NaN,triệu VNĐ,Thoả thuận,2025-06-23,2025-07-10,"[tiếng anh, giao tiếp]","[giao tiếp, tiếng anh]","giao tiếp, tiếng anh"
2,Tài Xế Lái Xe Văn Phòng (Nam) - Từ 05 Năm Kinh...,CÔNG TY CỔ PHẦN GIẢI PHÁP ĐIỆN TỬ VIỆT HỒNG QUANG,10.0,15.0,triệu VNĐ,Khoảng,2025-06-23,2025-07-10,[],[Bằng Lái Xe B2],Bằng Lái Xe B2
3,Chuyên Viên Kinh Doanh Không Yêu Cầu Kinh Nghi...,CÔNG TY TNHH BẤT ĐỘNG SẢN HỒNG ĐỨC LAND,16.0,NaN,triệu VNĐ,Tối thiểu,2025-06-23,2025-07-13,[],"[Kiên Trì, Đam mê kinh doanh]","Kiên Trì, Đam mê kinh doanh"
4,Nhân Viên Kinh Doanh/ Sale Supervisor Lĩnh Vưc...,CÔNG TY TNHH Y TẾ QUỐC TẾ MAIA,NaN,NaN,triệu VNĐ,Thoả thuận,2025-06-23,2025-07-09,"[excel, làm việc nhóm, giao tiếp]","[excel, giao tiếp, làm việc nhóm]","excel, giao tiếp, làm việc nhóm"
5,Chuyên Viên Kinh Doanh Trực Tuyến Thẻ (Telesales),Ngân Hàng Quốc tế (VIB),10.0,17.0,triệu VNĐ,Khoảng,2025-06-23,2025-07-09,"[tiếng anh, giao tiếp]","[Kỹ Năng Xử Lý Phản Hồi Khách Hàng, Kỹ năng Th...","Kỹ Năng Xử Lý Phản Hồi Khách Hàng, Kỹ năng Thu..."
6,"Nhân Viên Vận Hành Sàn TMĐT (TikTok, Shoppe) -...",CÔNG TY TNHH MOGGA VIỆT NAM,9.0,22.0,triệu VNĐ,Khoảng,2025-06-23,2025-07-20,"[phân tích, giao tiếp]","[giao tiếp, phân tích]","giao tiếp, phân tích"
7,Phiên Dịch Tiếng Trung (Thu Nhập Hấp Dẫn Từ 9 ...,CÔNG TY CỔ PHẦN POCKET CINE,9.0,15.0,triệu VNĐ,Khoảng,2025-06-23,2025-07-09,[excel],[excel],excel
8,Nhân Viên Kinh Doanh/ Sale Bất Động Sản Nhà Ph...,CÔNG TY CP BẤT ĐỘNG SẢN HECOLAND,20.0,100.0,triệu VNĐ,Khoảng,2025-06-23,2025-07-19,"[phân tích, giao tiếp]","[Direct Sale, Khả năng thuyết phục, Kinh Doanh...","Direct Sale, Khả năng thuyết phục, Kinh Doanh ..."
9,"Chỉ Huy Trưởng, Chỉ Huy Phó Thi Công Cơ Điện (...",Công ty Cổ phần Đầu tư Thương mại và Kỹ thuật ...,25.0,35.0,triệu VNĐ,Khoảng,2025-06-23,2025-07-15,"[phân tích, giao tiếp, ba]","[C++, Giải quyết vấn đề, Hiểu Biết Về Bản Vẽ K...","C++, Giải quyết vấn đề, Hiểu Biết Về Bản Vẽ Kỹ..."



--- Thông tin của DataFrame đã xử lý ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   job_id                     50 non-null     int64         
 1   job_title                  50 non-null     object        
 2   job_url                    50 non-null     object        
 3   scrape_date                50 non-null     object        
 4   status                     50 non-null     object        
 5   company_name_raw_list      50 non-null     object        
 6   salary_raw_list            50 non-null     object        
 7   location_raw_list          50 non-null     object        
 8   post_date_raw_list         50 non-null     object        
 9   company_name_detail        50 non-null     object        
 10  company_scale              50 non-null     object        
 11  company_field              50 n